# Momants CSV-sentimentprocessor

Deze notebook verwerkt een Momants CSV-export met `tabularisai/multilingual-sentiment-analysis`. Er zit geen voorbeelddata in het project: vul hieronder alleen het pad naar je eigen CSV in.

De processor gebruikt `conversation_id` om berichten te groeperen en `created_at` om ze binnen ieder gesprek chronologisch te ordenen. Alleen bezoekersberichten (`from_agent == False`) worden geclassificeerd.

## 1. De processor importeren

De eigenlijke verwerking staat in `momants_sentiment.py`. Daardoor kun je dezelfde code vanuit deze notebook én vanaf de commandoregel gebruiken.

In [1]:
from pathlib import Path
import sys

PROJECTMAP = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECTMAP) not in sys.path:
    sys.path.insert(0, str(PROJECTMAP))

from momants_sentiment import laad_momants_csv, selecteer_bezoekersberichten, verwerk_csv

## 2. Kies de CSV en uitvoermap

Vervang het voorbeeldpad door het pad naar de Momants-export. De uitvoer bevat standaard geen oorspronkelijke berichttekst.

In [2]:
CSV_PAD = PROJECTMAP / "attached_assets" / "test_gesprekken_100.csv"
UITVOERMAP = PROJECTMAP / "resultaten"

print(f"Invoer: {CSV_PAD}")
print(f"Uitvoer: {UITVOERMAP}")

Invoer: /home/runner/workspace/attached_assets/test_gesprekken_100.csv
Uitvoer: /home/runner/workspace/resultaten


## 3. Controleer eerst de structuur

Deze stap start het model nog niet. Hij controleert of de CSV kan worden ingelezen en laat alleen aantallen zien.

In [3]:
data = laad_momants_csv(CSV_PAD)
bezoekersberichten = selecteer_bezoekersberichten(data)

print(f"Berichtrijen: {len(data)}")
print(f"Bruikbare bezoekersberichten: {len(bezoekersberichten)}")
print(f"Gesprekken: {bezoekersberichten['conversation_id'].nunique()}")

Berichtrijen: 90044
Bruikbare bezoekersberichten: 38696
Gesprekken: 19817


## 4. Bepaal start- en eindsentiment

Deze stap laadt het TabularisAI-model en schrijft één tabel met het startmoment in de naam, bijvoorbeeld `sentiment_per_gesprek_20260901_143522_123456.csv`. Ieder gesprek krijgt apart het sentiment van het eerste en het laatste bruikbare bezoekersbericht. Bij één bezoekersbericht worden start en eind op dezelfde, één keer geclassificeerde modeluitkomst gebaseerd.

In [4]:
gesprekresultaten = verwerk_csv(
    csv_pad=CSV_PAD,
    uitvoermap=UITVOERMAP,
    batchgrootte=32,
)

print(f"Bestand: {gesprekresultaten.attrs['uitvoerpad']}")
gesprekresultaten.head(10)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,conversation_id,aantal_bezoekersberichten,sentiment_start,zekerheid_start,sentiment_eind,zekerheid_eind,uitleg
0,0005f4b8-4ce7-4df2-9daf-538a0dd034ef,2,Neutraal (taakgericht),0.9115,Neutraal (taakgericht),0.9112,Het gesprek begint neutraal (taakgericht) en e...
1,00088073-1099-4cc5-b006-20cf62acc55d,2,Neutraal (taakgericht),0.5977,Neutraal (taakgericht),0.8667,Het gesprek begint neutraal (taakgericht) en e...
2,00149bf6-63d7-42cb-8633-f316956c0251,1,Neutraal (taakgericht),0.8623,Neutraal (taakgericht),0.8623,Het gesprek begint neutraal (taakgericht) en e...
3,0014ca76-8df1-426c-93f7-e76e554066a7,2,Neutraal (taakgericht),0.9215,Neutraal (taakgericht),0.9116,Het gesprek begint neutraal (taakgericht) en e...
4,001bcf17-9fc6-4977-8121-2b0a075b2866,2,Neutraal (taakgericht),0.3278,Neutraal (taakgericht),0.7414,Het gesprek begint neutraal (taakgericht) en e...
5,001fcc28-bf70-4018-b58c-0c94d9368e64,1,Neutraal (taakgericht),0.8475,Neutraal (taakgericht),0.8475,Het gesprek begint neutraal (taakgericht) en e...
6,0021303c-3049-43e1-bf51-fddba62bb0a2,1,Neutraal (taakgericht),0.9375,Neutraal (taakgericht),0.9375,Het gesprek begint neutraal (taakgericht) en e...
7,0028c93e-68fe-4c4d-9536-562ffc2bdd86,1,Neutraal (taakgericht),0.4455,Neutraal (taakgericht),0.4455,Het gesprek begint neutraal (taakgericht) en e...
8,00299fa6-8016-4123-b40c-1e8bc8bf03bb,3,Neutraal (taakgericht),0.8677,Negatief (gefrustreerd),0.4798,Het gesprek begint neutraal (taakgericht) en e...
9,002a0da4-d34c-451b-9e7a-0e3a0d2a8c20,1,Boos (paniek),0.5579,Boos (paniek),0.5579,Het gesprek begint boos (paniek) en eindigt bo...
